In [1]:
import os, pickle
import pandas as pd

In [2]:
with open("../../../training_data/7.Extra_set/features.pkl", "rb") as f:
    featuresd = pickle.load(f)

len(featuresd), featuresd

(30,
 {'21du':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       21du               5             E          264            R   
  1       21du               5             E          265            R   
  2       21du               5             E          266            R   
  3       21du               5             E          267            R   
  4       21du               5             E          268            R   
  ..       ...             ...           ...          ...          ...   
  273     21du               5             E          546            R   
  274     21du               5             E          547            R   
  275     21du               5             E          548            R   
  276     21du               5             E          549            R   
  277     21du               5             E          550            R   
  
                      

In [3]:
with open("../../../training_data/8.Apos/Extra_set/final_apos.pkl", "rb") as f:
    apos = pickle.load(f)
    
len(apos), apos

(24,
 {'8jp0': '8sgj',
  '8f4s': '7l6r',
  '7gqu': '6yhr',
  '7yg5': '7xlq',
  '8v81': '5uak',
  '6z1m': '6z1h',
  '9mfs': '8szq',
  '7f8p': '5e1i',
  '7p2v': '6gx3',
  '7qpn': '2gk9',
  '7u4k': '2gs3',
  '7v39': '7v37',
  '7zpe': '9di9',
  '8cgw': '5c97',
  '8crc': '2ojx',
  '8fpi': '6uen',
  '8qtk': '3pfv',
  '8y6w': '8y6y',
  '9dol': '4n22',
  '9ebs': '9di2',
  '9fsj': '9fsk',
  '9o2m': '5e97',
  '9oul': '9ouk',
  '9prs': '8yf0'})

In [4]:
print(featuresd.keys())

dict_keys(['21du', '22mj', '6s3a', '6vvq', '6z1m', '7e40', '7f8p', '7gqu', '7p2v', '7qpn', '7u4k', '7v39', '7yg5', '7zpe', '8cgw', '8crc', '8f4s', '8fpi', '8jp0', '8qtk', '8uk6', '8v81', '8y6w', '9dol', '9ebs', '9fsj', '9mfs', '9o2m', '9prs', '9oul'])


# "Predict"

On website.

AllositePro doesn't find any sites for **9ebs; 8fpi couldn't be run**

<br>

In [5]:
# Any missing
missing = []

for pdb in featuresd:
    pdbdir = f"{pdb}"
    
    if os.path.isdir(f"{pdbdir}_fail") or os.path.isdir(f"{pdbdir}_cannotrun"):
        print("Failed:", pdb)
    elif not os.path.isdir(pdbdir):
    #     os.makedirs(pdbdir, exist_ok=True)
    # elif len(os.listdir(pdbdir)) < 2:
    #     pass
    # else:
    #     continue
        missing.append(pdb)
        print("Missing:", pdb)


len(missing), sorted(missing)

Missing: 21du
Missing: 7e40
Missing: 7f8p
Missing: 7u4k
Missing: 7v39
Missing: 8cgw
Failed: 8fpi
Missing: 8qtk
Missing: 8y6w
Missing: 9dol
Failed: 9ebs
Missing: 9o2m
Missing: 9prs


(11,
 ['21du',
  '7e40',
  '7f8p',
  '7u4k',
  '7v39',
  '8cgw',
  '8qtk',
  '8y6w',
  '9dol',
  '9o2m',
  '9prs'])

In [6]:
missing_wapo = [m for m in missing if m in apos]

len(missing_wapo), sorted(missing_wapo)

(9, ['7f8p', '7u4k', '7v39', '8cgw', '8qtk', '8y6w', '9dol', '9o2m', '9prs'])

In [8]:
existing = [p for p in featuresd if p not in missing]

print(len(existing))
print(existing)

19
['22mj', '6s3a', '6vvq', '6z1m', '7gqu', '7p2v', '7qpn', '7yg5', '7zpe', '8crc', '8f4s', '8fpi', '8jp0', '8uk6', '8v81', '9ebs', '9fsj', '9mfs', '9oul']


In [4]:
import tarfile

In [5]:
# Uncompress manually placed result
for pdb in featuresd:
    pdbdir = f"{pdb}"
    if not os.path.isdir(pdbdir):
        print("Missing result .tar.gz:", pdb)
        continue
    tar_exists = False
    for f in os.listdir(pdbdir):
        if f.endswith(".tar.gz"):
            tar_exists = True
            outdir = f"{pdbdir}/{f.replace(f'{pdb}_', '').replace('.tar.gz', '')}"
            if not os.path.isdir(outdir):
                with tarfile.open(f"{pdbdir}/{f}", 'r:gz') as tar:
                    tar.extractall(path=pdbdir)
                
    if not tar_exists:
        print("Missing result .tar.gz:", pdb)

Missing result .tar.gz: 21du
Missing result .tar.gz: 7e40
Missing result .tar.gz: 7f8p
Missing result .tar.gz: 7u4k
Missing result .tar.gz: 7v39
Missing result .tar.gz: 8cgw
Missing result .tar.gz: 8fpi
Missing result .tar.gz: 8qtk
Missing result .tar.gz: 8y6w
Missing result .tar.gz: 9dol
Missing result .tar.gz: 9ebs
Missing result .tar.gz: 9o2m
Missing result .tar.gz: 9prs


In [6]:
from Bio import PDB

In [7]:
# Sanity check
for pdb in featuresd:
    pdbdir = f"{pdb}"
    if not os.path.isdir(pdbdir):
        print("Skipping", pdb)
        continue
        
    pdb_exists = False
    for d in os.listdir(pdbdir):
        if os.path.isdir(f"{pdbdir}/{d}") and not d.startswith("."):
            pdbf = f"{pdbdir}/{d}/{d.replace('_download', '')}.pdb"
            if os.path.isfile(pdbf):
                # Check that the length of the input pdb and output pdb (without heteroatoms of the pocket spheres) are the same
                if len(tuple(
                    residue
                    for model in PDB.PDBParser(QUIET=True).get_structure(pdb, f"../structures/{pdb}.pdb")
                    for chain in model
                    for residue in chain
                )) != len(tuple(
                    residue
                    for model in PDB.PDBParser(QUIET=True).get_structure(pdb, pdbf)
                    for chain in model
                    for residue in chain
                    if residue.id[0] == ' ' # it's not heteroatom                    
                )):
                    print("Input and output PDBs with different number of residues:", pdb)
            else:
                print("Missing pdb:", pdb)

Skipping 21du


Skipping 7e40
Skipping 7f8p


Skipping 7u4k
Skipping 7v39


Skipping 8cgw


Skipping 8fpi
Skipping 8qtk


Skipping 8y6w
Skipping 9dol
Skipping 9ebs


Skipping 9o2m
Skipping 9prs


# Processing

In [8]:
def process_allositepro_txt(file):
    with open(file, "r") as f:
        text = f.read().strip()
        lines = text.split('\n')
        pockets = {}
        current_pocket = None
    
        for line in lines:
            if line.startswith('pocket'):
                current_pocket = line
                pockets[current_pocket] = {}
            else:
                try:
                    key, value = line.split(':')
                except:
                    print(line)
                pockets[current_pocket][key.strip()] = float(value.strip())
    
        return pd.DataFrame(pockets).T.to_dict(orient="index")#.sort_values("hitScore", ascending=False)

In [9]:
import re

def process_allositepro_pml(file):
    with open(file, "r") as f:
        pymol_text = f.read().strip()
        pockets_dict = {}

        ## Lines can look like either:
        ## cmd.select("pocket2","///A/569+619+495+493+534+301+491+567+553+")
        ## cmd.select("pocket0","///A/187.0+15.0+250.0+251.0+186.0+247.0+254.0+") # PDBs with insertion codes
        
        # Pattern to match each pocket selection command
        pocket_pattern = re.compile(r'cmd\.select\("(?P<pocket>pocket\d+)",".+?"\)')
        # Pattern to match chain and residues within each selection command
        chain_residue_pattern = re.compile(r'///(?P<chain>\w+)/(?P<residues>[\d\.\+]+)')
    
        for pocket_match in pocket_pattern.finditer(pymol_text):
            pocket = pocket_match.group('pocket')
            selection_command = pocket_match.group(0)
            residues_list = []

            for chain_residue_match in chain_residue_pattern.finditer(selection_command):
                chain = chain_residue_match.group('chain')
                residues = chain_residue_match.group('residues').split('+')
                
                if any(res.endswith(".0") for res in residues): # Residues of PDBs with insertion codes
                    residues = tuple(res.replace(".0", "") for res in residues)

                residues_list.extend([(res, chain) for res in residues if res.isdigit()])
    
            if pocket not in pockets_dict:
                pockets_dict[pocket] = {
                    'auth_asym_id': [],
                    'auth_seq_id': []
                }
    
            for res, chain in residues_list:
                pockets_dict[pocket]['auth_seq_id'].append(res)
                pockets_dict[pocket]['auth_asym_id'].append(chain)
    
        return pockets_dict, chain_residue_match

In [10]:
allositepro_results = {}

for pdb in featuresd:
    pdbdir = f"{pdb}"
    if not os.path.isdir(pdbdir) or len([d for d in os.listdir(pdbdir) if not d.startswith(".")]) == 0:
        print("Missing:", pdb)
        continue

    jobid = next(f for f in os.listdir(pdbdir) if f.endswith(".tar.gz")).replace(f"{pdb}_", "").replace("_download.tar.gz", "")
    d = f"{pdbdir}/{jobid}_download"

    pockets = process_allositepro_txt(f"{d}/{jobid}.txt")
    pockets_res, chain_residue_match = process_allositepro_pml(f"{d}/{jobid}.pml")
    assert set(pockets.keys()) == set(pockets_res.keys())
    for p in list(pockets.keys()):
        pockets[p]["residues"] = pd.DataFrame(pockets_res[p], dtype=str)

    allositepro_results[pdb] = pockets

len(allositepro_results), allositepro_results

Missing: 21du
Missing: 7e40
Missing: 7f8p
Missing: 7u4k
Missing: 7v39
Missing: 8cgw
Missing: 8fpi
Missing: 8qtk
Missing: 8y6w
Missing: 9dol
Missing: 9ebs
Missing: 9o2m
Missing: 9prs


(17,
 {'22mj': {'pocket0': {'Volume': 992.587,
    'SASA': 508.691,
    'Druggability Score': 0.868,
    'logitProb': 0.822,
    'nmaScore': 0.196,
    'hitScore': 0.697,
    'residues':    auth_asym_id auth_seq_id
    0             A         164
    1             A         163
    2             A         150
    3             A         146
    4             A         195
    5             A         193
    6             A         118
    7             A         133
    8             A         223
    9             A         153
    10            A         151
    11            A         131
    12            A         272
    13            A         160
    14            A         148
    15            A         167
    16            A         236
    17            A         116
    18            A         189
    19            A         119
    20            A         120
    21            A         222
    22            A         233
    23            A         132
    24           

In [11]:
allositepro_resultsf = "allositepro_results.pkl"

with open(allositepro_resultsf, "wb") as f:
    pickle.dump(allositepro_results, f)